# Carga de datos

Este notebook **no forma parte del flujo normal de creación de un modelo**: en producción, los datos ya deberían estar materializados como una tabla en el DWH/Datalake de la empresa, resultado de un proceso de ingesta independiente. Aquí se simula ese origen usando un dataset de ejemplo no productivo en `.csv` (`BD_creditos.csv`).

El único propósito de este notebook es dejar la fuente de datos y su validación básica de esquema documentadas en un solo lugar (`config.json`). Por ahora, `heuristic_model.py` ya lo lee (rutas y opciones de lectura); `comprension_eda.ipynb` y `ft_engineering.py` todavía tienen sus propias rutas hardcodeadas y no consumen este archivo — migrarlos queda pendiente para una siguiente iteración, para no romper su comportamiento actual en este PR.

In [ ]:
import json
from pathlib import Path

import pandas as pd

## Configuración de la fuente

Se centraliza en `config.json` (a la par de este notebook, en `src/`) la ruta del dataset crudo y el esquema de columnas esperado. Si en el futuro la fuente cambia (por ejemplo, a una consulta contra el Datalake), solo hay que actualizar este archivo, sin tocar el resto de los notebooks/scripts que ya usan estas rutas.

In [ ]:
RUTA_CONFIG = Path('config.json')

config = {
    'raw_data_path': '../BD_creditos.csv',
    'curated_data_path': '../BD_creditos_modelo.csv',
    'curated_data_pkl_path': '../BD_creditos_modelo.pkl',
    'csv_read_options': {
        # Cada archivo tiene su propio formato: el crudo usa separador ';'
        # y coma decimal; el curado (exportado por comprension_eda.ipynb
        # con pandas.to_csv) usa los defaults de pandas (',' y '.'). Usar
        # las mismas opciones para ambos rompe la lectura del curado
        # (colapsa sus columnas en una sola).
        'raw': {
            'sep': ';',
            'decimal': ',',
            'encoding': 'utf-8-sig',
        },
        'curated': {
            'sep': ',',
            'decimal': '.',
        },
    },
    'columnas_esperadas': [
        'tipo_credito', 'fecha_prestamo', 'capital_prestado', 'plazo_meses',
        'edad_cliente', 'tipo_laboral', 'salario_cliente', 'total_otros_prestamos',
        'cuota_pactada', 'puntaje', 'puntaje_datacredito', 'cant_creditosvigentes',
        'huella_consulta', 'saldo_mora', 'saldo_total', 'saldo_principal',
        'saldo_mora_codeudor', 'creditos_sectorFinanciero', 'creditos_sectorCooperativo',
        'creditos_sectorReal', 'promedio_ingresos_datacredito', 'tendencia_ingresos',
        'Pago_atiempo',
    ],
    # Esquema del dataset curado (BD_creditos_modelo.csv): sin 'saldo_principal'
    # (eliminado por redundancia en el EDA) y con 'relacion_cuota_salario'
    # (derivada agregada por ft_engineering.py).
    'columnas_esperadas_curado': [
        'tipo_credito', 'fecha_prestamo', 'capital_prestado', 'plazo_meses',
        'edad_cliente', 'tipo_laboral', 'salario_cliente', 'total_otros_prestamos',
        'cuota_pactada', 'puntaje_datacredito', 'cant_creditosvigentes',
        'huella_consulta', 'saldo_mora', 'saldo_total',
        'saldo_mora_codeudor', 'creditos_sectorFinanciero', 'creditos_sectorCooperativo',
        'creditos_sectorReal', 'promedio_ingresos_datacredito', 'tendencia_ingresos',
        'Pago_atiempo', 'relacion_cuota_salario',
    ],
    'target_col': 'Pago_atiempo',
}

RUTA_CONFIG.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Configuración escrita en {RUTA_CONFIG.resolve()}')

## Carga y validación básica

Se carga el CSV con las mismas opciones documentadas en `config.json` y se valida que el esquema (columnas presentes) coincida con lo esperado, antes de pasar el dataset a la siguiente etapa (`comprension_eda.ipynb`).

In [ ]:
config = json.loads(RUTA_CONFIG.read_text(encoding='utf-8'))

df = pd.read_csv(config['raw_data_path'], **config['csv_read_options']['raw'])
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
columnas_esperadas = set(config['columnas_esperadas'])
columnas_actuales = set(df.columns)

faltantes = columnas_esperadas - columnas_actuales
inesperadas = columnas_actuales - columnas_esperadas

if faltantes:
    raise ValueError(f'Faltan columnas esperadas en la fuente: {sorted(faltantes)}')
if inesperadas:
    print(f'Advertencia: columnas nuevas no documentadas en config.json: {sorted(inesperadas)}')

print('Esquema validado: todas las columnas esperadas están presentes.')
print(f"\nNulos por columna:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Con la fuente cargada y el esquema validado, este dataframe (`df`) queda listo para ser consumido por las siguientes etapas. Por ahora `comprension_eda.ipynb` sigue haciendo su propia carga con una ruta hardcodeada (no lee `config.json` todavía); la migración a leer desde aquí queda pendiente para no acoplar ambos notebooks en esta iteración.